# NiHAN Nesting Optimizer — Benchmark
**GPU:** T4 (Colab ücretsiz) veya A100 (Colab Pro)

## Setup

In [ ]:
# 1. Repo'yu clone et
!git clone https://github.com/keremyavuz25-alt/nihan-nesting-optimizer.git
%cd nihan-nesting-optimizer

In [ ]:
# 2. Bağımlılıkları yükle
!pip install -q ezdxf shapely matplotlib
# torch zaten Colab'da yüklü

In [ ]:
# 3. GPU kontrol
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('GPU yok — CPU mode')

## Hız Testi — CPU vs GPU

In [ ]:
# 4. Hız testi
!python test_speed.py

## Benchmark — 8 Algoritma

### CPU BLF (baseline)

In [ ]:
# 5a. CPU BLF benchmark (kısa — 500 iter, pop=50)
!python benchmark.py test.dxf 1500 blf 50 500 1

### GPU Benchmark

In [ ]:
# 5b. GPU benchmark (kısa — 500 iter, pop=50)
!python benchmark.py test.dxf 1500 gpu 50 500 1

In [ ]:
# 5c. GPU benchmark (orta — 2000 iter, pop=500)
!python benchmark.py test.dxf 1500 gpu 500 2000 1

In [ ]:
# 5d. GPU benchmark (uzun — 5000 iter, pop=2000) — ~30dk-1saat
!python benchmark.py test.dxf 1500 gpu 2000 5000 1

## Sonuçları Görüntüle

In [ ]:
# 6. Sonuçları oku
import json, glob
for f in sorted(glob.glob('results_*/benchmark_report.json')):
    with open(f) as fp:
        data = json.load(fp)
    print(f'\n=== {data["decoder"].upper()} ===')
    print(f'Pop: {data["pop_size"]}, Iter: {data["max_iter"]}, Wall: {data["wall_clock_seconds"]:.0f}s')
    for r in data['results']:
        print(f'  {r["algorithm"]:<25} {r["best"]:>7.2f}%  ({r["avg_time"]:.1f}s)')

In [ ]:
# 7. SVG görselleştirme (en iyi sonuç)
from IPython.display import SVG, display
import glob
svgs = sorted(glob.glob('results_gpu/*.svg'))
if svgs:
    print(f'{len(svgs)} SVG dosyası:')
    for s in svgs:
        print(f'  {s}')
        display(SVG(filename=s))
else:
    svgs = sorted(glob.glob('results_blf/*.svg'))
    for s in svgs[:3]:
        display(SVG(filename=s))

## Optimal Batch Size Testi

In [ ]:
# 8. Farklı batch size'ları dene
from dxf_parser import load_dxf
from gpu_decoder import GPUDecoder
import numpy as np, time

pieces = load_dxf('test.dxf')
n = len(pieces)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dec = GPUDecoder(pieces, bin_width=1500, resolution=3.0, device=device)

for B in [50, 100, 500, 1000, 2000, 5000, 10000]:
    try:
        np.random.seed(42)
        seqs = [np.random.permutation(n).tolist() for _ in range(B)]
        rots = [np.random.uniform(0, 360, size=n).tolist() for _ in range(B)]
        
        # Warmup
        _ = dec.batch_fitness(seqs[:10], rots[:10])
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        
        t0 = time.time()
        fits = dec.batch_fitness(seqs, rots)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.time() - t0
        
        vram = torch.cuda.memory_allocated()/1e6 if torch.cuda.is_available() else 0
        print(f'B={B:>6d}: {elapsed:>7.3f}s  avg={elapsed/B*1000:.3f}ms/decode  VRAM={vram:.0f}MB  fitness=[{min(fits):.1f}, {max(fits):.1f}]%')
    except RuntimeError as e:
        print(f'B={B:>6d}: OOM — {e}')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        break